# Reconstructing Supplementary Figures
This notebook can be used to reproduce Supplementary Figures from the gVAMP paper.
## Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

## Size and font

In [ ]:
eps = 1e-32

img_width = 16
img_height = 16

fs_normal = 18
fs_large = 20
fs_small = 16
fw_sublbl = 500
fw_axlbl = 500

## Source data
Specify the path to the source data for the figures here

In [ ]:
FigS1_source_fpath = "FigS1_source.csv"
FigS2_source_fpath = "FigS2_source.csv"
FigS3_source_fpath = "FigS3_source.csv"
FigS4_source_fpath = "FigS4_source.csv"
FigS5_source_fpath = "FigS5_source.csv"
FigS6_source_fpath = "FigS6_source.csv"
FigS7_source_fpath = "FigS7_source.csv"
FigS8_source_fpath = "FigS8_source.csv"
FigS9_source_fpath = "FigS9_source.csv"
FigS10_source_fpath = "FigS10_source.csv"

## Supplementary Figure 1

In [ ]:
# S1
df = pd.read_table(FigS1_source_fpath, sep="\t")

ncols=4
nrows=3

colors = ['tab:blue', 'tab:green']

labels = df["Label"].unique()
subplts = df["Subplot"].unique()

tpr_lims = [0.6, 0.6, 0.5, 0.5, 0.5, 0.4, 0.4, 0.3, 0.3, 0.4, 0.2, 0.3]
fdr_lims = [0.5, 0.4, 0.4, 0.4, 0.3, 0.1, 0.4, 0.1, 0.2, 0.1, 0.2, 0.1]
locators = [0.1, 0.1, 0.1, 0.1, 0.1, 0.05, 0.1, 0.05, 0.1 ,0.05, 0.1, 0.05]

fig, ax = plt.subplots(nrows, ncols, figsize=(16,12))

for k,s in enumerate(subplts):
    i = int(k / ncols)
    j = k - i * ncols

    for lbl_idx, lbl in enumerate(labels):
        tpr_mean = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["TPR"].values
        fdr_mean = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["FDR"].values
        ax[i,j].plot(fdr_mean, tpr_mean, alpha=0.5, color=colors[lbl_idx % len(colors)], label=lbl)
        tpr_std_upper = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_upper")]["TPR"].values
        tpr_std_lower = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_lower")]["TPR"].values
        ax[i,j].fill_between(fdr_mean, tpr_std_lower, tpr_std_upper, color="grey", alpha=0.2)

        mean_tpr_95 = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "point")]["TPR"].values
        mean_fdr_95 = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "point")]["FDR"].values
        ax[i,j].scatter(mean_fdr_95, mean_tpr_95, color=colors[lbl_idx % len(colors)])       

    ax[i,j].set_title(subplts[k], fontsize=fs_normal, loc='left', fontweight=fw_sublbl)
    ax[i,j].set_xlim([0, fdr_lims[k]])
    #ax[i,j].set_ylim([0, tpr_lims[k]])
    ax[i,j].set_ylim([0,0.6])
    ax[i,j].vlines(0.05, ymin=0, ymax=1, color="black", linestyles="dashed", label="FDR05", alpha=0.5)
    ax[i,j].spines[["right", "top"]].set_visible(False)
    ax[i,j].tick_params(axis='x', labelsize=fs_normal)
    ax[i,j].tick_params(axis='y', labelsize=fs_normal)

ax[0,0].legend(loc="lower right", fontsize=fs_normal, frameon=False)
fig.supxlabel('False discovery rate', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('True positive rate', fontsize=fs_large, fontweight=fw_axlbl)

fig.tight_layout()
fig.savefig("FigS1.png", dpi=300)
plt.show()

## Supplementary Figure 2

In [ ]:
# S2
df = pd.read_table(FigS2_source_fpath, sep="\t")

ncols=4
nrows=3

colors = ['tab:blue', 'tab:green']

labels = df["Label"].unique()
subplts = df["Subplot"].unique()

fig, axs = plt.subplots(nrows, ncols, figsize=(16,12))

for k, s in enumerate(subplts):
    row, col = divmod(k, ncols)
    ax = axs[row, col]

    for lbl_idx, lbl in enumerate(labels):

        mean_fdr = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["FDR"].values
        thresholds = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["PIP_threshold"].values

        fdr_std_upper = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_upper")]["FDR"].values
        fdr_std_lower = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_lower")]["FDR"].values
        
        ax.plot(thresholds, mean_fdr, "-", label=lbl, color=colors[lbl_idx % len(colors)], alpha=0.5)
        ax.fill_between(thresholds, fdr_std_lower, fdr_std_upper, color="grey", alpha=0.2)

    ax.plot(thresholds, 1 - thresholds, linestyle='--', color='gray', zorder=0)
    ax.set_title(s, fontsize=fs_normal, loc='left', fontweight=fw_sublbl)
    ax.spines[["right", "top"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=fs_normal)
    ax.tick_params(axis='y', labelsize=fs_normal)
    ax.xaxis.set_major_locator(MultipleLocator(0.1))
    
    if col == 0 and row < (nrows-1):
        ax.spines[["bottom"]].set_visible(False)
        ax.set_xticklabels([])
        
    elif col > 0 and row < (nrows -1):
        ax.spines[["left", "bottom"]].set_visible(False)
        ax.set_yticklabels([])
        ax.set_xticklabels([])
    elif col > 0 and row == (nrows - 1):
        ax.spines[["left"]].set_visible(False)
        ax.set_yticklabels([])
        
    ax.grid(alpha=0.25)

    ax.set_xlim([0.5, 0.95])
    ax.set_ylim([0, 1])

axs[0,0].legend(loc="best", fontsize=fs_normal, frameon=False)
fig.supxlabel('PIP threshold', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('False discovery rate', fontsize=fs_large, fontweight=fw_axlbl)
fig.tight_layout()
fig.savefig("FigS2.png", dpi=300)
plt.show()

## Supplementary Figure 3

In [ ]:
# S3
df = pd.read_table(FigS3_source_fpath, sep="\t")

fig, ax = plt.subplots(2, 1, figsize=(16, 6), sharey=False)

width = 0.2

xlabels = df["Simulation_ID"].unique()
labels = df["Label"].unique()
x = np.arange(1, len(xlabels)+1)

plt.subplots_adjust(hspace=0.4)

method = "gVAMP"
fdr_mean = df[ (df["Label"] == method) & (df["Property"] == "mean")]["FDR"].values
tpr_mean = df[ (df["Label"] == method) & (df["Property"] == "mean")]["TPR"].values
tpr_std = df[ (df["Label"] == method) & (df["Property"] == "std")]["TPR"].values
fdr_std = df[(df["Label"] == method) & (df["Property"] == "std")]["FDR"].values
ax[0].bar(x - width/2, tpr_mean, width, yerr=tpr_std, label=method, color="tab:blue", alpha=0.5)
ax[1].bar(x - width/2, fdr_mean, width, yerr=fdr_std, label=method, color="tab:blue", alpha=0.5)

method = "FINEMAP"
fdr_mean = df[(df["Label"] == method) & (df["Property"] == "mean")]["FDR"].values
tpr_mean = df[(df["Label"] == method) & (df["Property"] == "mean")]["TPR"].values
tpr_std = df[(df["Label"] == method) & (df["Property"] == "std")]["TPR"].values
fdr_std = df[(df["Label"] == method) & (df["Property"] == "std")]["FDR"].values
ax[0].bar(x + width/2, tpr_mean, width, yerr=tpr_std, label=method, color="tab:green", alpha=0.5)
ax[1].bar(x + width/2, fdr_mean, width, yerr=fdr_std, label=method, color="tab:green", alpha=0.5)

ax[0].set_ylabel('TPR', fontsize=fs_large, fontweight=fw_axlbl)
ax[0].set_xticks(x)
ax[0].set_xticklabels(xlabels)
ax[0].set_ylim([0, 1])
ax[0].yaxis.set_tick_params(labelsize=fs_normal)
ax[0].spines[["right", "top"]].set_visible(False)
ax[0].legend(fontsize=fs_normal, frameon=False, loc="upper left")
ax[0].set_xticklabels([]) 
ax[0].set_xlim([0.5, np.max(x) + 0.5])
ax[0].tick_params(axis='y', labelsize=fs_normal)

ax[1].set_xlabel('simulation ID', fontsize=fs_large, fontweight=fw_axlbl)
ax[1].set_xticks(x)
ax[1].set_ylabel('FDR', fontsize=fs_large, fontweight=fw_axlbl)
ax[1].hlines(0.05, xmin=0, xmax=np.max(x)+0.5, color="black", linestyles="dashed", alpha=1)
ax[1].yaxis.set_tick_params(labelsize=fs_normal)
ax[1].spines[["right", "top"]].set_visible(False)
ax[1].set_ylim([0, 1])
ax[1].set_xlim([0.5, np.max(x) + 0.5])

ax[1].set_xticklabels(xlabels, fontsize=fs_normal)
ax[1].tick_params(axis='y', labelsize=fs_normal)

plt.tight_layout()
plt.savefig('FigS3.png', dpi=300, bbox_inches='tight')
plt.show()

## Supplementary Figure 4

In [ ]:
# S4
df = pd.read_table(FigS4_source_fpath, sep="\t")

ncols=2
nrows=2

colors = [ 'tab:blue','tab:orange','tab:green',  'tab:red', 'black']

labels = df["Label"].unique()
subplts = df["Subplot"].unique()

tpr_lims = [0.6, 0.6, 0.5, 0.4]
fdr_lims = [0.5, 0.4, 0.3, 0.1]

fig, ax = plt.subplots(nrows, ncols, figsize=(10,8))

for k,s in enumerate(subplts):
    i = int(k / ncols)
    j = k - i * ncols

    for lbl_idx, lbl in enumerate(labels):
        tpr_mean = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["TPR"].values
        fdr_mean = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "mean")]["FDR"].values
        ax[i,j].plot(fdr_mean, tpr_mean, alpha=0.5, color=colors[lbl_idx % len(colors)], label=lbl)
        tpr_std_upper = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_upper")]["TPR"].values
        tpr_std_lower = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "std_lower")]["TPR"].values
        ax[i,j].fill_between(fdr_mean, tpr_std_lower, tpr_std_upper, color="grey", alpha=0.2)

        mean_tpr_95 = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "point")]["TPR"].values
        mean_fdr_95 = df[(df["Subplot"] == s) & (df["Label"] == lbl) & (df["Property"] == "point")]["FDR"].values
        ax[i,j].scatter(mean_fdr_95, mean_tpr_95, color=colors[lbl_idx % len(colors)]) 
      

    ax[i,j].set_title(f"Simulation setting {s}", fontsize=fs_normal, fontweight=fw_sublbl)
    ax[i,j].set_xlim([0, fdr_lims[k]])
    ax[i,j].set_ylim([0, tpr_lims[k]])
    ax[i,j].xaxis.set_tick_params(labelsize=fs_normal)
    ax[i,j].yaxis.set_tick_params(labelsize=fs_normal)
    ax[i,j].vlines(0.05, ymin=0, ymax=1, color="black", linestyles="dashed", alpha=0.5)
    ax[i,j].spines[["right", "top"]].set_visible(False)
    ax[i,j].tick_params(axis='x', labelsize=fs_normal)
    ax[i,j].tick_params(axis='y', labelsize=fs_normal)

fig.supxlabel('False discovery rate', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('True positive rate', fontsize=fs_large, fontweight=fw_axlbl)
ax[0,1].legend(fontsize=fs_normal, frameon=False, ncols=1, bbox_to_anchor=(1, 1), loc="upper left")
fig.tight_layout()
fig.savefig("FigS4.png", dpi=300)
plt.show()

## Supplementary Figure 5

In [ ]:
#S5
df = pd.read_table(FigS5_source_fpath, sep="\t")

bar_width = 0.1
colors = ['tab:green', 'tab:blue', 'tab:orange', 'tab:red', 'tab:purple']
offset_y = 0.01

bin_labels = df["MAF_bin"].unique()
x = np.arange(len(bin_labels))
labels = df["Label"].unique()
subplots = df["Subplot"].unique()
n_rows = 6
n_cols = 2
fig, axs = plt.subplots(n_rows, n_cols, figsize=(13, 3 * n_rows), sharex=True)
axs = axs.flatten()

for i in range(len(subplots)):
    ax = axs[i]
    subplt_lbl = chr(ord('a') + i)
    for lbl_idx, lbl in enumerate(labels):
        means = df[(df["Subplot"] == subplt_lbl) & (df["Label"] == lbl) & (df["Property"] == "mean")]["Proportion_of_variance"].values
        stds = df[(df["Subplot"] == subplt_lbl) & (df["Label"] == lbl) & (df["Property"] == "std")]["Proportion_of_variance"].values
        
        if lbl == "Ground_truth":

            offset = (2 - len(labels) / 2) * bar_width
            
            ax.bar(
                x + offset,
                means,
                yerr=stds,
                width=bar_width,
                color='none',
                edgecolor='red',
                linewidth=2,
                label='Ground truth',
                capsize=3,
                zorder=3,
                error_kw=dict(ecolor='red', lw=1.5)
            )
            
        else:
            offset = (lbl_idx - len(labels) / 2) * bar_width

            ax.bar(x + offset,
                   means, 
                   yerr=stds, 
                   width=bar_width,
                   color=colors[lbl_idx % len(colors)], 
                   alpha=0.5,
                   label=lbl,
                   capsize=3
                  )

    if i % n_cols == 0:
        ax.set_ylabel('Proportion of Variance', fontsize=10)

    def format_bin_label(start, end):
        def fmt(x):
            return f"{x:.4f}".rstrip('0').rstrip('.') if x < 0.01 else f"{x:.3f}".rstrip('0').rstrip('.')
        return f"{fmt(start)}–{fmt(end)}"

    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels, rotation=45)
    ax.set_ylim(0, 1)
    ax.spines[['right', 'top']].set_visible(False)
    ax.grid(alpha=0.1)

    if i == 0:
        ax.legend(fontsize=8, loc='upper right')

for col in range(n_cols):
    axs[-n_cols + col].set_xlabel('MAF bins', fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.subplots_adjust(hspace=0.5, wspace=0.3)
plt.tight_layout()
fig.savefig("FigS5.png", dpi=300)
plt.show()

## Supplementary Figure 6

In [ ]:
#S6
df = pd.read_table(FigS6_source_fpath, sep="\t")

bar_width = 0.1
colors = ['tab:green', 'tab:blue', 'tab:orange', 'tab:red', 'tab:purple']
offset_y = 0.01

bin_labels = df["MAF_bin"].unique()
x = np.arange(len(bin_labels))
rows = df["Row"].unique().astype(int)
labels = df["Label"].unique()

n_rows = len(rows)

fig, axs = plt.subplots(n_rows, 1, figsize=(img_width, img_height), sharex=True)

for i in rows:
    ax = axs[i]
    for lbl_idx, lbl in enumerate(labels):
        means = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "mean")]["Proportion_of_variance"].values
        stds = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "std")]["Proportion_of_variance"].values
        
        if lbl == "Ground_truth":
            offset = (2 - len(labels) / 2) * bar_width
            
            ax.bar(
                x + offset,
                means,
                yerr=stds,
                width=bar_width,
                color='none',
                edgecolor='red',
                linewidth=2,
                label='Ground truth',
                capsize=3,
                zorder=3,
                error_kw=dict(ecolor='red', lw=1.5)
            )
            
        else:
            offset = (lbl_idx - len(labels) / 2) * bar_width

            ax.bar(x + offset,
                   means, 
                   yerr=stds, 
                   width=bar_width,
                   color=colors[lbl_idx % len(colors)], 
                   alpha=0.5,
                   label=lbl,
                   capsize=3
                  )

    ax.tick_params(axis='x', labelsize=fs_normal)
    ax.tick_params(axis='y', labelsize=fs_normal)
    ax.set_ylim([0,1])
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels)
    ax.spines[['right', 'top']].set_visible(False)
    if i < n_rows:
        ax.spines['bottom'].set_visible(False)
    ax.grid(alpha=0.1)

axs[0].legend(fontsize=fs_normal, loc='upper right', frameon=False, ncol=2)
fig.supxlabel('MAF bins', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('Proportion of variance', fontsize=fs_large, fontweight=fw_axlbl)
plt.tight_layout()
fig.savefig("FigS6.png", dpi=300)
plt.show()

## Supplementary Figure 7

In [ ]:
#S7
df = pd.read_table(FigS7_source_fpath, sep="\t")

colors = {"coding": "#66c2a5", "noncoding": "#fc8d62"}
bar_width = 0.15
gap_between_groups = 0.5
type_offset = {"True": 0, "Est": bar_width*2}  # offset Est bars

fig, ax = plt.subplots(figsize=(14,6))
x_ticks = []
x_tick_labels = []

categories = df["Category"].unique()
hatches = df["Hatch"].unique()
positions = df["Pos"].unique()

for pos in positions:
    df_ = df[(df["Pos"] == pos)]
    mean_val = df_['Mean']
    sd_val = df_['Standard_deviation']
    hatch = str(df_["Hatch"].values[0])
    cat = df_["Category"].values[0]
    typ = df_["Type"].values[0]

    ax.bar(pos, mean_val, width=bar_width, yerr=sd_val,
                       color=colors[cat], edgecolor="black", hatch=hatch,
                       capsize=3, label=f"{typ} {cat}")

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc="upper left", frameon=True)

ax.set_ylabel("effect size")
plt.tight_layout()

fig.savefig("FigS7.png", dpi=300, bbox_inches='tight')
plt.show()

## Supplementary Figure 8

In [ ]:
#S8
df = pd.read_table(FigS8_source_fpath, sep="\t")

bin_labels = df["MAF_bin"].unique()
x_positions = np.arange(len(bin_labels))

for i in x_positions:
    r2s = df[df["MAF_bin"] == bin_labels[i]]["Correlation_squared"].values
    plt.boxplot(r2s, positions=[i], showmeans=True, widths=0.5)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.grid()     
plt.xticks(x_positions, bin_labels, rotation=90, fontsize=10)
plt.xlabel("MAF bins", fontsize=12)
plt.ylabel("Correlation squared", fontsize=12)
plt.tight_layout()
plt.savefig("FigS8.png", dpi=300)
plt.show()

## Supplementary Figure 9

In [ ]:
#S9
df = pd.read_table(FigS9_source_fpath, sep="\t")

rows = df["Row"].unique()
cols = df["Column"].unique()

xlim = [-0.05, 0.07]
ylim = [-0.03, 0.03]

ncols=2
nrows=3

fig, axs = plt.subplots(nrows, ncols, figsize=(12, 16))

fig.supylabel('gVAMP estimates', fontsize=fs_large, fontweight=fw_axlbl)
fig.supxlabel('Marginal estimates', fontsize=fs_large, fontweight=fw_axlbl)

df_source = pd.DataFrame({})

for i in rows:
    for j in cols:
        ax = axs[i,j]

        x = df[(df["Row"] == i) & (df["Column"] == j)]["Marginal_estimates"].values
        y = df[(df["Row"] == i) & (df["Column"] == j)]["gVAMP_estimates"].values
        ax.scatter(x, y, alpha=0.5, s=1)
        ax.spines[["right", "top"]].set_visible(False)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.tick_params(axis='x', labelsize=fs_normal)
        ax.tick_params(axis='y', labelsize=fs_normal)

        if j > 0:
            ax.spines[["right", "top", "left"]].set_visible(False)
            ax.set_yticklabels([])
        if i < (nrows - 1):
            ax.spines[["bottom"]].set_visible(False)
            ax.set_xticklabels([])

axs[0,0].set_title("Common", fontsize=fs_normal, weight="bold")
axs[0,1].set_title("Rare", fontsize=fs_normal, weight="bold")

axs[0,0].set_ylabel("Imputed SNPs - height", fontsize=fs_normal, weight="bold")
axs[1,0].set_ylabel("WGS variants - height", fontsize=fs_normal, weight="bold")
axs[2,0].set_ylabel("WGS variants - simulation", fontsize=fs_normal, weight="bold")

plt.tight_layout()
plt.savefig('FigS9.png', dpi=300, bbox_inches='tight')
plt.show()

## Supplementary Figure 10

In [ ]:
#S10
df = pd.read_table(FigS10_source_fpath, sep="\t")

ukb_coefs = df["UKB_gVAMP_coefficients"]
allofus_coefs = df["AllofUs_marginal_coefficients"]

plt.figure(figsize=(8,8))
plt.plot(ukb_coefs, allofus_coefs, "o", color="tab:blue", alpha=0.5)
plt.plot([-1, 1], [-1, 1], color='gray', linestyle='--', linewidth=2, alpha=0.5)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.xlim([-0.03, 0.03])
plt.ylim([-0.03, 0.03])
plt.xlabel("gVAMP UKB joint regression coefficients", fontsize=fs_large)
plt.ylabel("AllOfUs marginal regression coefficients", fontsize=fs_large)
plt.tick_params(axis='y', labelsize=fs_normal)
plt.tick_params(axis='x', labelsize=fs_normal)
plt.tight_layout()

plt.savefig("FigS10.png", dpi=300)
plt.show()